<a href="https://colab.research.google.com/github/PriyankaDevaprasad/Forest-Cover-Type-Classification/blob/main/notebook/Forest_cover_streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 96.9 MB/s eta 0:00:00
--2026-10-05 08:11:27--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64 [following]
--2026-10-05 08:11:27--  https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/c98ec3ed-1305-47d9-91a6-a895afc557b4?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-05T08%3A51%3A28Z&rscd=attachment%3B+filename%3D

In [5]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import pickle


model = joblib.load("best_model.pkl")
scaler = joblib.load("scaler.pkl")
cover_encoder = joblib.load("Cover_Type.pkl")

st.set_page_config(page_title="Forest Cover Type Predictor",page_icon="🌲",layout="wide")
st.title("🌲 Forest Cover Type Predictor")
st.markdown(
    "<p style='color:green; font-size:25px;'>Predict the forest cover type using terrain and environmental features.</p>",
    unsafe_allow_html=True)

st.divider()
st.subheader("Enter the Terrain Features")
col1,col2=st.columns(2)
with col1:
  Elevation=st.number_input("Elevation",key="Elevation")
  Horizontal_Distance_To_Roadways=st.number_input("Horizontal_Distance_To_Roadways",key="Horizontal_Distance_To_Roadways")
  Horizontal_Distance_To_Fire_Points=st.number_input("Horizontal_Distance_To_Fire_Points",key="Horizontal_Distance_To_Fire_Points")
  Soil_Type=st.selectbox("Soil_Type",list(range(1, 41)))
  Wilderness_Area=st.selectbox("Wilderness_Area",['1','2','3','4'])
with col2:
  Distance_To_Hydrology=st.number_input("Distance_To_Hydrology",key="Distance_To_Hydrology")
  Aspect=st.number_input("Aspect",key="Aspect")
  Mean_Hillshade=st.number_input("Mean_Hillshade",key="Mean_Hillshade")
  predict=st.button("Predict the forest type", type="primary",key="predict")

if predict:
  input_data = pd.DataFrame({
        "Elevation": [Elevation],
        "Horizontal_Distance_To_Roadways": [Horizontal_Distance_To_Roadways],
        "Horizontal_Distance_To_Fire_Points": [Horizontal_Distance_To_Fire_Points],
        "Soil_Type": [Soil_Type],
        "Wilderness_Area": [Wilderness_Area],
        "Distance_To_Hydrology": [Distance_To_Hydrology],
        "Aspect": [Aspect],
        "Mean_Hillshade": [Mean_Hillshade]
    })

  scale_cols = [
    "Elevation",
    "Horizontal_Distance_To_Roadways",
    "Horizontal_Distance_To_Fire_Points",
    "Distance_To_Hydrology",
    "Aspect",
    "Mean_Hillshade"]

  cols = ["Soil_Type", "Wilderness_Area"]

  input_data[cols] = input_data[cols].astype(int)

  # Apply the same scaling used during training
  input_data[scale_cols] = scaler.transform(input_data[scale_cols])

  prediction = model.predict(input_data)
  predicted_class = cover_encoder.inverse_transform(prediction.astype(int))
  st.markdown(
        f"""
        <div style="
            background-color:#66C5CC;
            padding:20px;
            border-radius:10px;
            color:#000000;
            font-size:22px;
            font-weight:bold;
            margin-top:20px;
        ">
            🌲 Predicted Forest Cover Type: {predicted_class[0]}
        </div>
        """,
        unsafe_allow_html=True)


Overwriting app.py


In [6]:
!streamlit run /content/app.py &>/content/logs.txt &

In [7]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://promises-since-gradually-identification.trycloudflare.com
